### Title: 02a_build_ingredient_hierarchy
### Purpose: creates a hierarchical taxonomy for NHANES ingredients similar to the DietDiveR Food Tree structure, with hierarchical codes and a mapping from original ingredient codes to the new hierarchical system
### Author: Jules Larke
### Date: November 07, 2025

### Import packages

In [ ]:
import pandas as pd
import re
from collections import defaultdict, OrderedDict

### Main script for categorizing ingredients into food levels based on conditional statments and direct assignment

In [ ]:
def load_ingredients():
    """Load and categorize ingredients"""
    ingredients = pd.read_csv('../../data/00/food_tree/nhanes_unique_ingredients.tsv', sep='\t')
    
    def categorize_ingredient(code, desc):
        """Categorize ingredient based on description with priority order"""
        desc_lower = desc.lower()
        
        # Priority 1: Leavening agents / yeast
        if 'leavening' in desc_lower or (desc_lower.startswith('yeast') and 'extract' not in desc_lower):
            return 9, 'Leavening_Additives'
        
        # Priority 2: Butter goes to Fats/Oils (for dietary substitution comparisons)
        if desc_lower.startswith('butter,'):
            return 7, 'Fats_Oils'
        
        # Priority 3: Specific spices and salt (ONLY "Salt, table")
        if desc_lower == 'salt, table':
            return 9, 'Seasonings_Condiments'
        if desc_lower.startswith('spices,') or 'spice' in desc_lower:
            return 9, 'Seasonings_Condiments'
        # Sauce but NOT hot peppers with sauce
        if any(word in desc_lower for word in ['vinegar']) and 'vanilla' not in desc_lower:
            return 9, 'Seasonings_Condiments'
        if 'sauce,' in desc_lower and 'pepper' not in desc_lower and 'applesauce' not in desc_lower:
            return 9, 'Seasonings_Condiments'
        
        # Priority 4: Vegetable oil should be fats, not vegetables
        if desc_lower.startswith('vegetable oil'):
            return 7, 'Fats_Oils'
        
        # Priority 5: Animal fats (Fat, chicken, Fat, turkey, etc.)
        if desc_lower.startswith('fat,') or 'animal fat' in desc_lower:
            return 7, 'Fats_Oils'
        
        # Priority 6: Milk and dairy (but not butter)
        if any(word in desc_lower for word in ['milk', 'cheese', 'cream', 'yogurt', 'whey', 'dulce de leche']):
            if 'peanut butter' not in desc_lower and 'almond butter' not in desc_lower:
                # Exception: "Cream of wheat" is a grain, not dairy
                if 'cream of wheat' in desc_lower:
                    return 4, 'Grains'
                # Exception: Coconut milk and coconut cream are nut products, not dairy
                if 'coconut' in desc_lower:
                    return 3, 'Legumes_Nuts_Seeds'
                # Exception: Potato products that mention "milk" (e.g., "without milk") are vegetables
                if 'potato' in desc_lower:
                    return 6, 'Vegetables'
                return 1, 'Milk_Products'
        
        # Priority 6b: Alcoholic beverages (before general beverages)
        if any(alc in desc_lower for alc in ['alcoholic beverage', 'wine,', 'beer', 'liqueur', 'sake', 'distilled']):
            return 8, 'Beverages'
        
        # Priority 7: Beverages (coffee, tea) - but NOT "water pack" foods or "steamed" foods
        # IMPORTANT: Check for coffee and tea BEFORE checking for water to avoid misclassification
        if 'coffee' in desc_lower or 'tea,' in desc_lower:
            # Exception: coffee liqueur is alcohol, not coffee
            if 'liqueur' in desc_lower:
                return 8, 'Beverages'
            # Exception: steamed foods are not tea
            if ', cooked, steamed' not in desc_lower:
                return 8, 'Beverages'

        # Hard cider should be in beverages/alcohol
        if 'hard cider' in desc_lower:
            return 8, 'Beverages'

        # Check if this is just a food "in water" or "water pack" - should NOT be classified as beverage
        if ', water pack,' in desc_lower or ', canned in water,' in desc_lower:
            # This is a food preserved in water, not a water beverage - skip to other categories
            pass
        elif desc_lower.startswith('beverages,'):
            # Exception: beverages that are alcohol
            if 'liqueur' in desc_lower or 'alcohol' in desc_lower or 'wine' in desc_lower or 'beer' in desc_lower:
                return 8, 'Beverages'
            return 8, 'Beverages'
        elif desc_lower.startswith('water,'):
            return 8, 'Beverages'
        
        # Priority 8: Meat, Poultry, Fish, Eggs
        if any(word in desc_lower for word in ['egg,', 'beef', 'pork', 'lamb', 'veal', 'chicken', 'turkey', 
                                                'duck', 'goose', 'quail', 'pheasant', 'dove', 'fish,', 
                                                'salmon', 'tuna', 'catfish', 'trout', 'cod', 'halibut',
                                                'herring', 'sardine', 'anchovy', 'tilapia', 'haddock',
                                                'pollock', 'pompano', 'mullet', 'perch', 'pike',
                                                'crustaceans', 'shrimp', 'crab', 'lobster', 'crayfish',
                                                'mollusks', 'oyster', 'clam', 'mussel', 'squid', 'octopus',
                                                'abalone', 'scallop', 'snail', 'game meat',
                                                'bacon', 'ham', 'sausage', 'variety meats',
                                                'pork skins']):
            # Exception: coconut meat is a nut, not animal meat
            if 'coconut' in desc_lower:
                return 3, 'Legumes_Nuts_Seeds'
            # Exception: meatless products
            if 'meatless' in desc_lower or 'meat substitute' in desc_lower:
                return 3, 'Soy_Products'
            # Exception: Fat, chicken is a fat, not chicken meat
            if desc_lower.startswith('fat,'):
                return 7, 'Fats_Oils'
            return 2, 'Meat_Poultry_Fish_Eggs'
        
        # Priority 8b: Amphibians (frog, turtle) - separate from fish/meat
        # BUT exclude "black turtle beans"
        if ('frog' in desc_lower or 'turtle' in desc_lower) and 'black turtle' not in desc_lower and 'beans' not in desc_lower:
            return 2, 'Meat_Poultry_Fish_Eggs'
        
        # Priority 9: Snap beans and edible-podded peas are vegetables, not legumes
        if 'beans, snap' in desc_lower or 'edible-podded' in desc_lower:
            return 6, 'Vegetables'
        
        # Priority 10: Legumes, Nuts, Seeds (includes coconut)
        if any(word in desc_lower for word in ['beans,', 'bean,', 'peas, green', 'peas, split', 'pea,', 'lima', 'lentils', 'chickpea', 'cowpea', 'pigeonpea', 'broadbean',
                                                'soy', 'tofu', 'miso', 'carob', 'coconut']):
            return 3, 'Legumes_Nuts_Seeds'
        
        if any(word in desc_lower for word in ['nuts,', 'peanut', 'almond', 
                                                'cashew', 'walnut', 'pecan', 'pistachio', 'hazelnut',
                                                'macadamia', 'brazil nut', 'pine nut', 'chestnut',
                                                'brazilnuts']):
            # Exception: waterchestnuts are vegetables, not nuts
            if 'waterchestnut' not in desc_lower:
                return 3, 'Legumes_Nuts_Seeds'
        
        if desc_lower.startswith('seeds,') or desc_lower.startswith('seed,'):
            return 3, 'Legumes_Nuts_Seeds'
        
        # Priority 11: Grains
        if any(word in desc_lower for word in ['flour', 'wheat', 'rice', 'oat', 'barley', 'rye',
                                                'corn', 'cereal', 'grain', 'millet', 'quinoa',
                                                'cornmeal', 'grits', 'popcorn', 'hominy', 'wild rice',
                                                'buckwheat', 'tapioca', 'babyfood, cereal']):
            # Exception: corn syrup goes to sugars
            if 'syrup' in desc_lower:
                return 8, 'Sugars_Sweets'
            # Exception: potato flour goes to vegetables
            if 'potato flour' in desc_lower:
                return 6, 'Vegetables'
            # Exception: peanut flour goes to legumes
            if 'peanut flour' in desc_lower:
                return 3, 'Legumes_Nuts_Seeds'          
            return 4, 'Grains'
        
        # Priority 12: Fruits (includes fruit juices, but NOT avocado, olive)
        fruit_keywords = ['apple', 'apple juice', 'apricot', 'banana', 'cherry', 'cherries', 'grape', 'grapefruit',
                         'orange', 'peach', 'pear', 'plum', 'prune', 'strawberr', 'blueberr',
                         'raspberr', 'blackberr', 'cranberr', 'boysenberr', 'lemon', 'lime',
                         'mango', 'pineapple', 'melon', 'watermelon', 'cantaloupe', 'honeydew',
                         'fig', 'date', 'raisin', 'currant', 'papaya', 'guava', 'kiwi',
                         'persimmon', 'nectarine', 'tangerine', 'pomegranate',
                         'plantain', 'passion-fruit', 'litchi', 'lychee', 'kumquat',
                         'loganberr', 'mulberr', 'rhubarb', 'tamarind', 'fruit,', 'carambola']
        
        # Exclude balsam-pear and waxgourd (they're vegetables, not fruits)
        if any(word in desc_lower for word in fruit_keywords) and 'balsam-pear' not in desc_lower and 'waxgourd' not in desc_lower:
            return 5, 'Fruits'
        
        # Priority 13: Vegetables (includes avocado, olive, seaweed, lotus, waxgourd, balsam-pear)
        veg_keywords = ['potato', 'tomato', 'lettuce', 'carrot', 'celery', 'onion', 'pepper',
                       'cabbage', 'broccoli', 'spinach', 'mushroom', 'cucumber', 'squash',
                       'pumpkin', 'kale', 'asparagus', 'beet', 'cauliflower', 'okra',
                       'radish', 'turnip', 'parsley', 'garlic', 'ginger', 'eggplant',
                       'zucchini', 'collard', 'mustard green', 'chard', 'artichoke',
                       'brussels sprout', 'cress', 'dandelion', 'endive', 'escarole',
                       'leek', 'kohlrabi', 'parsnip', 'rutabaga', 'salsify', 'celeriac',
                       'chive', 'fennel', 'arugula', 'radicchio', 'watercress',
                       'tomatillo', 'nopal', 'cassava', 'yam', 'taro', 'jicama',
                       'bamboo', 'hearts of palm', 'seaweed', 'vegetable', 'pimento',
                       'burdock', 'balsam-pear', 'avocado', 'olive', 'lotus', 'waxgourd',
                       'waterchestnut', 'pokeberry', 'alfalfa', 'capers']
        
        if any(word in desc_lower for word in veg_keywords):
            return 6, 'Vegetables'
        
        # Priority 14: Fats and Oils (includes animal fats)
        if any(word in desc_lower for word in ['oil', 'lard', 'shortening', 'grease', 'ghee']):
                # Exception: items with 'boiled' in the description
                if 'boiled' not in desc_lower:
                    return 7, 'Fats_Oils'
        
        # Priority 15: Sugars and Sweets (includes gelatin)
        if any(word in desc_lower for word in ['sugar', 'honey', 'syrup', 'molasses', 'sweetener',
                                               'chocolate', 'cocoa', 'gelatin', 'snacks,']):
            return 8, 'Sugars_Sweets'
        
        # Default - herbs and other seasonings
        if any(herb in desc_lower for herb in ['basil', 'oregano', 'thyme', 'parsley', 'cilantro', 
                                               'coriander', 'sage', 'marjoram']):
            return 9, 'Seasonings_Condiments'
        
        # Default
        return 9, 'Other'
    
    # Apply categorization
    ingredients['main_code'] = ingredients.apply(
        lambda row: categorize_ingredient(row['ingred_code'], row['ingred_desc'])[0], axis=1
    )
    ingredients['main_category'] = ingredients.apply(
        lambda row: categorize_ingredient(row['ingred_code'], row['ingred_desc'])[1], axis=1
    )
    
    return ingredients


def build_hierarchy(ingredients_df):
    """
    Build the hierarchical taxonomy structure
    Returns: OrderedDict with hierarchical structure
    """
    
    hierarchy = OrderedDict()
    
    # Define main categories (Level 1)
    main_categories = {
        1: 'Milk_and_Milk_Products',
        2: 'Meat_Poultry_Fish_and_Eggs',
        3: 'Legumes_Nuts_and_Seeds',
        4: 'Grain_Products',
        5: 'Fruits_and_Fruit_Products',
        6: 'Vegetables_and_Vegetable_Products',
        7: 'Fats_Oils',
        8: 'Sugars_Sweets_and_Beverages',
        9: 'Seasonings_Condiments_and_Other_Ingredients'
    }
    
    # Add main categories
    for code, name in main_categories.items():
        hierarchy[str(code)] = name
    
    # Now build subcategories for each main category
    # This is where we create the 2-digit, 3-digit, etc. levels
    
    # ========================================================================
    # CATEGORY 1: MILK AND MILK PRODUCTS
    # ========================================================================
    
    # Level 2 (2-digit)
    hierarchy['11'] = 'Fluid_milks'
    hierarchy['12'] = 'Dried_milks_and_milk_powders'
    hierarchy['13'] = 'Creams'
    hierarchy['14'] = 'Cheeses'
    hierarchy['15'] = 'Other_milk_products'
    
    # Level 3 (3-digit) and beyond
    # Fluid milks
    hierarchy['111'] = 'Milk_fluid_whole'
    hierarchy['112'] = 'Milk_fluid_reduced_fat'
    hierarchy['113'] = 'Milk_fluid_low_fat'
    hierarchy['114'] = 'Milk_fluid_nonfat'
    hierarchy['115'] = 'Buttermilk_fluid'
    hierarchy['116'] = 'Goat_milk'
    hierarchy['117'] = 'Other_fluid_milks'
    
    # Dried milks
    hierarchy['121'] = 'Milk_dried_whole'
    hierarchy['122'] = 'Milk_dried_nonfat'
    hierarchy['123'] = 'Milk_dried_other'
    hierarchy['124'] = 'Buttermilk_dried'
    
    # Creams
    hierarchy['131'] = 'Cream_fluid'
    hierarchy['132'] = 'Half_and_half'
    
    # Cheeses
    hierarchy['141'] = 'Cheese_natural'
    hierarchy['142'] = 'Cheese_cottage_and_ricotta'
    hierarchy['143'] = 'Cheese_cream'
    hierarchy['144'] = 'Cheese_Mexican_types'
    hierarchy['145'] = 'Cheese_reduced_fat'
    
    # Mexican cheese subtypes
    hierarchy['1441'] = 'Cheese_queso_fresco_and_cotija'
    hierarchy['1442'] = 'Cheese_oaxaca_and_chihuahua'
    hierarchy['1443'] = 'Cheese_queso_seco_and_anejo'
    
    # Other
    hierarchy['151'] = 'Whey_products'
    hierarchy['152'] = 'Evaporated_and_condensed_milk'
    hierarchy['153'] = 'Sour_cream'
    hierarchy['154'] = 'Other_dairy_products'
    
    # ========================================================================
    # CATEGORY 2: MEAT, POULTRY, FISH, AND EGGS
    # ========================================================================
    
    hierarchy['21'] = 'Beef'
    hierarchy['22'] = 'Pork'
    hierarchy['23'] = 'Poultry'
    hierarchy['24'] = 'Fish'
    hierarchy['25'] = 'Shellfish'
    hierarchy['26'] = 'Game_meats'
    hierarchy['27'] = 'Organ_meats'
    hierarchy['28'] = 'Eggs'
    hierarchy['29'] = 'Lamb'
    
    # Beef subcategories
    hierarchy['211'] = 'Beef_ground'
    hierarchy['212'] = 'Beef_cuts_cooked'
    hierarchy['213'] = 'Beef_processed'
    hierarchy['214'] = 'Veal'
    
    # Pork
    hierarchy['221'] = 'Pork_fresh'
    hierarchy['222'] = 'Ham'
    hierarchy['223'] = 'Bacon'
    
    # Poultry
    hierarchy['231'] = 'Chicken'
    hierarchy['232'] = 'Turkey'
    hierarchy['233'] = 'Other_poultry'
    hierarchy['2331'] = 'Duck'
    hierarchy['2332'] = 'Game_birds'
    
    # Fish
    hierarchy['241'] = 'Fish_fresh_or_frozen'
    hierarchy['242'] = 'Fish_canned'
    hierarchy['243'] = 'Fish_smoked_or_cured'
    
    # Shellfish
    hierarchy['251'] = 'Crustaceans'
    hierarchy['252'] = 'Mollusks'
    hierarchy['2521'] = 'Clams_and_mussels'
    hierarchy['2522'] = 'Oysters'
    hierarchy['2523'] = 'Squid_and_octopus'
    hierarchy['2524'] = 'Other_mollusks'
    
    # Game meats
    hierarchy['261'] = 'Deer_and_venison'
    hierarchy['262'] = 'Other_game'
    
    # Amphibians
    hierarchy['263'] = 'Amphibians'
    
    # Organ meats
    hierarchy['271'] = 'Liver'
    hierarchy['272'] = 'Other_organs'
    hierarchy['272'] = 'Other_organs'
    
    # Eggs
    hierarchy['281'] = 'Eggs_whole'
    hierarchy['282'] = 'Egg_whites'
    hierarchy['283'] = 'Egg_yolks'
    hierarchy['284'] = 'Other_eggs'
    
    # ========================================================================
    # CATEGORY 3: LEGUMES, NUTS, AND SEEDS
    # ========================================================================
    
    hierarchy['31'] = 'Legumes'
    hierarchy['32'] = 'Nuts'
    hierarchy['33'] = 'Seeds'
    hierarchy['34'] = 'Soy_products'
    
    # Legumes
    hierarchy['311'] = 'Beans'
    hierarchy['312'] = 'Peas'
    hierarchy['313'] = 'Lentils'
    hierarchy['314'] = 'Chickpeas_and_other_legumes'
    
    hierarchy['3111'] = 'Beans_kidney'
    hierarchy['3112'] = 'Beans_pinto'
    hierarchy['3113'] = 'Beans_black'
    hierarchy['3114'] = 'Beans_white_navy_great_northern'
    hierarchy['3115'] = 'Beans_lima'
    hierarchy['3116'] = 'Beans_other'
    hierarchy['3117'] = 'Carob'
    
    # Peas subcategories
    hierarchy['3121'] = 'Peas_green'
    hierarchy['3122'] = 'Peas_split'
    hierarchy['3123'] = 'Peas_other'
    
    # Nuts
    hierarchy['321'] = 'Peanuts'
    hierarchy['322'] = 'Tree_nuts'
    hierarchy['323'] = 'Coconut_products'
    
    hierarchy['3221'] = 'Almonds'
    hierarchy['3222'] = 'Walnuts'
    hierarchy['3223'] = 'Pecans'
    hierarchy['3224'] = 'Cashews'
    hierarchy['3225'] = 'Brazil_nuts'
    
    # Seeds
    hierarchy['331'] = 'Sunflower_seeds'
    hierarchy['332'] = 'Pumpkin_seeds'
    hierarchy['333'] = 'Sesame_seeds'
    hierarchy['334'] = 'Other_seeds'
    
    # Soy products
    hierarchy['341'] = 'Soybeans'
    hierarchy['342'] = 'Tofu_and_soy_curd'
    hierarchy['343'] = 'Soy_flour_and_protein'
    hierarchy['344'] = 'Other_soy_products'
    
    # ========================================================================
    # CATEGORY 4: GRAIN PRODUCTS
    # ========================================================================
    
    hierarchy['41'] = 'Wheat_products'
    hierarchy['42'] = 'Rice_products'
    hierarchy['43'] = 'Corn_products'
    hierarchy['44'] = 'Oat_products'
    hierarchy['45'] = 'Other_grains'
    hierarchy['46'] = 'Cereal_products'
    
    # Wheat
    hierarchy['411'] = 'Wheat_flour'
    hierarchy['412'] = 'Wheat_bran_and_germ'
    
    # Rice
    hierarchy['421'] = 'Rice_white'
    hierarchy['422'] = 'Rice_brown'
    hierarchy['423'] = 'Rice_flour'
    hierarchy['424'] = 'Wild_rice'
    
    # Corn
    hierarchy['431'] = 'Corn_flour_and_meal'
    hierarchy['432'] = 'Corn_grits'
    hierarchy['433'] = 'Cornstarch'
    hierarchy['434'] = 'Popcorn'
    hierarchy['435'] = 'Hominy'
    
    # Oats
    hierarchy['441'] = 'Oat_flour'
    hierarchy['442'] = 'Oat_bran'
    hierarchy['443'] = 'Oatmeal'
    
    # Other grains
    hierarchy['451'] = 'Barley'
    hierarchy['452'] = 'Rye'
    hierarchy['453'] = 'Millet'
    hierarchy['454'] = 'Quinoa'
    hierarchy['455'] = 'Buckwheat'
    hierarchy['456'] = 'Tapioca'
    hierarchy['457'] = 'Other'
    
    # Cereals
    hierarchy['461'] = 'Ready_to_eat_cereals'
    hierarchy['462'] = 'Hot_cereals'
    hierarchy['463'] = 'Baby_food_cereals'
    
    # ========================================================================
    # CATEGORY 5: FRUITS AND FRUIT PRODUCTS
    # ========================================================================
    
    hierarchy['51'] = 'Citrus_fruits'
    hierarchy['52'] = 'Berries'
    hierarchy['53'] = 'Pome_fruits'  # Apples, pears
    hierarchy['54'] = 'Stone_fruits'  # Peaches, cherries, plums, apricots
    hierarchy['55'] = 'Tropical_fruits'
    hierarchy['56'] = 'Melons'  
    hierarchy['57'] = 'Grapes'  
    hierarchy['58'] = 'Dried_fruits'
    hierarchy['59'] = 'Other_fruits'
    
    # Citrus
    hierarchy['511'] = 'Oranges_and_tangerines'
    hierarchy['512'] = 'Grapefruit'
    hierarchy['513'] = 'Lemons_and_limes'
    hierarchy['514'] = 'Other_citrus'
    
    # Berries
    hierarchy['521'] = 'Strawberries'
    hierarchy['522'] = 'Blueberries'
    hierarchy['523'] = 'Raspberries'
    hierarchy['524'] = 'Blackberries'
    hierarchy['525'] = 'Cranberries'
    hierarchy['526'] = 'Other_berries'
    
    # Pome fruits
    hierarchy['531'] = 'Apples'
    hierarchy['532'] = 'Pears'
    hierarchy['533'] = 'Asian_pears'
    
    # Stone fruits
    hierarchy['541'] = 'Peaches_and_nectarines'
    hierarchy['542'] = 'Cherries'
    hierarchy['543'] = 'Plums'
    hierarchy['544'] = 'Apricots'
    hierarchy['545'] = 'Other_stone_fruits'
    
    # Tropical
    hierarchy['551'] = 'Bananas_and_plantains'
    hierarchy['552'] = 'Pineapple'
    hierarchy['553'] = 'Mango'
    hierarchy['554'] = 'Papaya'
    hierarchy['555'] = 'Other_tropical'
    
    # Melons
    hierarchy['561'] = 'Watermelon'
    hierarchy['562'] = 'Cantaloupe'
    hierarchy['563'] = 'Honeydew'
    hierarchy['564'] = 'Other_melons'
    
    # Grapes
    hierarchy['571'] = 'Grapes_fresh'
    hierarchy['572'] = 'Raisins_and_dried_grapes'
    
    # Dried fruits
    hierarchy['581'] = 'Dates_and_figs'
    hierarchy['582'] = 'Dried_apricots_peaches'
    hierarchy['583'] = 'Raisins_other_dried_fruits'
    
    # Other
    hierarchy['593'] = 'Kiwi'
    hierarchy['594'] = 'Pomegranate'
    hierarchy['595'] = 'Other'
    
    # ========================================================================
    # CATEGORY 6: VEGETABLES AND VEGETABLE PRODUCTS
    # ========================================================================
    
    hierarchy['61'] = 'Dark_green_vegetables'
    hierarchy['62'] = 'Deep_yellow_vegetables'
    hierarchy['63'] = 'Tomatoes_and_tomato_products'
    hierarchy['64'] = 'Potatoes'
    hierarchy['65'] = 'Other_starchy_vegetables'
    hierarchy['66'] = 'Other_vegetables'
    
    # Dark green
    hierarchy['611'] = 'Leafy_greens'
    hierarchy['612'] = 'Broccoli_and_broccoflower'
    hierarchy['613'] = 'Other_dark_green'
    hierarchy['6111'] = 'Spinach'
    hierarchy['6112'] = 'Kale_and_collards'
    hierarchy['6113'] = 'Lettuce_romaine'
    hierarchy['6114'] = 'Other_greens'
    
    # Deep yellow
    hierarchy['621'] = 'Carrots'
    hierarchy['622'] = 'Pumpkin'
    hierarchy['623'] = 'Sweet_potatoes'
    hierarchy['624'] = 'Winter_squash'
    
    # Tomatoes
    hierarchy['631'] = 'Tomatoes_raw'
    hierarchy['632'] = 'Tomatoes_cooked'
    hierarchy['633'] = 'Tomato_products_canned'
    hierarchy['6331'] = 'Tomato_paste'
    hierarchy['6332'] = 'Tomato_sauce'
    hierarchy['6333'] = 'Tomato_puree'
    
    # Potatoes
    hierarchy['641'] = 'Potatoes_fresh'
    hierarchy['642'] = 'Potatoes_canned'
    hierarchy['643'] = 'Potatoes_dehydrated'
    hierarchy['644'] = 'Potato_flour'
    
    # Other starchy
    hierarchy['654'] = 'Cassava_and_yuca'
    hierarchy['655'] = 'Taro_and_other_roots'
    
    # Other vegetables
    hierarchy['661'] = 'Other_vegetables'
    hierarchy['6610'] = 'Alliums'
    hierarchy['6611'] = 'Cucurbits'
    hierarchy['6612'] = 'Crucifers'
    hierarchy['6613'] = 'Mushrooms'
    hierarchy['6614'] = 'Snap_beans_and_edible_pods'
    hierarchy['6615'] = 'Peppers'
    hierarchy['6616'] = 'Seaweed'
    hierarchy['6617'] = 'Avocados'
    hierarchy['6618'] = 'Olives'
    
    hierarchy['66100'] = 'Onions'
    hierarchy['66101'] = 'Garlic'
    hierarchy['66102'] = 'Leeks_and_scallions'
    
    hierarchy['66110'] = 'Cucumbers'
    hierarchy['66111'] = 'Summer_squash'
    
    hierarchy['66120'] = 'Cabbage'
    hierarchy['66121'] = 'Cauliflower'
    hierarchy['66122'] = 'Brussels_sprouts'
    hierarchy['66123'] = 'Artichokes'
    hierarchy['66124'] = 'Asparagus'
    hierarchy['66125'] = 'Rutabaga'
    hierarchy['66126'] = 'Turnip'
    
    hierarchy['66150'] = 'Peppers_sweet'
    hierarchy['66151'] = 'Peppers_hot'

    hierarchy['662'] = 'Other_vegetables'

    hierarchy['6620'] = 'Tomatillos'
    
    # ========================================================================
    # CATEGORY 7: FATS AND OILS
    # ========================================================================
    
    hierarchy['71'] = 'Vegetable_oils'
    hierarchy['72'] = 'Animal_fats'
    hierarchy['73'] = 'Butter'
    
    # Vegetable oils
    hierarchy['711'] = 'General_vegetable_oils'
    hierarchy['712'] = 'Specific_oils'
    hierarchy['7121'] = 'Olive_oil'
    hierarchy['7122'] = 'Corn_oil'
    hierarchy['7123'] = 'Canola_oil'
    hierarchy['7124'] = 'Other_vegetable_oils'
    
    # Animal fats
    hierarchy['721'] = 'Pork_fat'
    hierarchy['722'] = 'Chicken_fat'
    hierarchy['723'] = 'Turkey_fat'
    hierarchy['724'] = 'Other_animal_fats'
    
    # ========================================================================
    # CATEGORY 8: SUGARS, SWEETS, AND BEVERAGES
    # ========================================================================
    
    hierarchy['80'] = 'Alcoholic_beverages'
    hierarchy['81'] = 'Sugars_and_sweeteners'
    hierarchy['82'] = 'Syrups'
    hierarchy['83'] = 'Sweet_toppings'
    hierarchy['84'] = 'Water_and_plain_beverages'
    hierarchy['85'] = 'Fruit_juices_and_nectars'
    hierarchy['87'] = 'Coffee_and_tea'
    hierarchy['88'] = 'Other_beverages'
    hierarchy['89'] = 'Cocoa_and_chocolate'
    
    # Alcoholic beverages
    hierarchy['801'] = 'Distilled_spirits'
    hierarchy['802'] = 'Wines'
    hierarchy['803'] = 'Beer_and_cider'
    hierarchy['804'] = 'Liqueurs'
    hierarchy['805'] = 'Sake'
    
    # Sugars
    hierarchy['811'] = 'White_sugar'
    hierarchy['812'] = 'Other_sugars'
    
    # Syrups
    hierarchy['821'] = 'Maple_syrup'
    hierarchy['822'] = 'Corn_syrup'
    #hierarchy['823'] = 'Honey' # creates duplicate labels for node and text description are identical
    #hierarchy['824'] = 'Molasses' # creates duplicate labels for node and text description are identical
    hierarchy['825'] = 'Agave_syrup'
    hierarchy['826'] = 'Other_syrups'
    
    # Gelatin
    hierarchy['830'] = 'Gelatin'
    
    # Water
    hierarchy['841'] = 'Tap_water'
    hierarchy['842'] = 'Bottled_water'
    
    # Coffee and tea
    hierarchy['871'] = 'Coffee'
    hierarchy['872'] = 'Tea'
    
    # Other beverages
    hierarchy['881'] = 'Whey_beverages'
    
    # Cocoa
    hierarchy['891'] = 'Cocoa_powder'
    hierarchy['892'] = 'Chocolate_baking'
    
    # ========================================================================
    # CATEGORY 9: SEASONINGS, CONDIMENTS, AND LEAVENING AGENTS
    # ========================================================================
    
    hierarchy['91'] = 'Salt_and_salt_products'
    hierarchy['92'] = 'Spices'
    hierarchy['93'] = 'Herbs'
    hierarchy['94'] = 'Vinegars'
    hierarchy['95'] = 'Sauces_and_condiments'
    hierarchy['96'] = 'Leavening_agents'
    hierarchy['97'] = 'Extracts_and_flavorings'
    
    # Spices
    hierarchy['921'] = 'Ground_spices_general'
    hierarchy['922'] = 'Seed_spices'
    
    # Herbs
    hierarchy['932'] = 'Fresh_herbs'
    
    # Sauces
    hierarchy['951'] = 'Hot_sauces'
    hierarchy['952'] = 'Other_sauces'
    
    # Leavening
    hierarchy['961'] = 'Yeast'
    hierarchy['962'] = 'Baking_powder'
    hierarchy['963'] = 'Baking_soda'
    hierarchy['964'] = 'Other_leavening'
    
    return hierarchy


def assign_hierarchical_codes(ingredients_df, hierarchy):
    """
    Assign hierarchical codes to each ingredient based on detailed classification
    """
    
    # Create a mapping dictionary for each ingredient
    code_mapping = []
    
    for _, row in ingredients_df.iterrows():
        orig_code = row['ingred_code']
        desc = row['ingred_desc']
        desc_lower = desc.lower()
        main_code = row['main_code']
        
        # Initialize with main category
        hier_codes = [str(main_code)]
        hier_labels = [hierarchy[str(main_code)]]
        
        # Now assign more specific codes based on detailed rules
        
        if main_code == 1:  # Milk and dairy
            hier_codes, hier_labels = classify_milk_product(desc_lower, hierarchy, hier_codes, hier_labels)
        
        elif main_code == 2:  # Meat, poultry, fish, eggs
            hier_codes, hier_labels = classify_meat_product(desc_lower, hierarchy, hier_codes, hier_labels)
        
        elif main_code == 3:  # Legumes, nuts, seeds
            hier_codes, hier_labels = classify_legume_nut_seed(desc_lower, hierarchy, hier_codes, hier_labels)
        
        elif main_code == 4:  # Grains
            hier_codes, hier_labels = classify_grain(desc_lower, hierarchy, hier_codes, hier_labels)
        
        elif main_code == 5:  # Fruits
            hier_codes, hier_labels = classify_fruit(desc_lower, hierarchy, hier_codes, hier_labels)
        
        elif main_code == 6:  # Vegetables
            hier_codes, hier_labels = classify_vegetable(desc_lower, hierarchy, hier_codes, hier_labels)
        
        elif main_code == 7:  # Fats and oils
            hier_codes, hier_labels = classify_fat_oil(desc_lower, hierarchy, hier_codes, hier_labels)
        
        elif main_code == 8:  # Sugars, sweets, beverages
            hier_codes, hier_labels = classify_sugar_beverage(desc_lower, hierarchy, hier_codes, hier_labels)
        
        elif main_code == 9:  # Seasonings, condiments
            hier_codes, hier_labels = classify_seasoning(desc_lower, hierarchy, hier_codes, hier_labels)
        
        # Create the final hierarchical code (most specific)
        final_hier_code = hier_codes[-1] if hier_codes else str(main_code)
        
        code_mapping.append({
            'original_code': orig_code,
            'ingredient_description': desc,
            'hierarchical_code': final_hier_code,
            'hierarchical_path': ' > '.join(hier_labels),
            'level': len(final_hier_code),
            'leaf_code': ''  # Will be filled in later
        })
    
    return pd.DataFrame(code_mapping)


# Classification functions for each category
def classify_milk_product(desc, hierarchy, codes, labels):
    """Classify milk products"""
    # CRITICAL: Check for cheese FIRST before milk to avoid misclassification
    # This prevents "Cheese, mozzarella, whole milk" from being classified as fluid milk
    if 'cheese' in desc:
        codes.append('14')
        labels.append(hierarchy['14'])
        # All natural cheeses go to one category
        if 'cottage' in desc or 'ricotta' in desc:
            codes.append('142')
            labels.append(hierarchy['142'])
        elif 'cream' in desc:
            codes.append('143')
            labels.append(hierarchy['143'])
        elif any(mex in desc for mex in ['mexican', 'queso', 'oaxaca', 'cotija', 'anejo', 'chihuahua', 'seco']):
            codes.append('144')
            labels.append(hierarchy['144'])
            if 'fresco' in desc or 'cotija' in desc:
                codes.append('1441')
                labels.append(hierarchy['1441'])
            elif 'oaxaca' in desc or 'chihuahua' in desc:
                codes.append('1442')
                labels.append(hierarchy['1442'])
            elif 'seco' in desc or 'anejo' in desc:
                codes.append('1443')
                labels.append(hierarchy['1443'])
        elif 'reduced fat' in desc or 'low fat' in desc or 'nonfat' in desc or 'fat free' in desc:
            codes.append('145')
            labels.append(hierarchy['145'])
        else:
            # All other natural cheeses (cheddar, mozzarella, swiss, parmesan, etc.)
            codes.append('141')
            labels.append(hierarchy['141'])
    
    elif 'whey' in desc:
        codes.extend(['15', '151'])
        labels.extend([hierarchy['15'], hierarchy['151']])
    
    elif 'cream' in desc and 'cheese' not in desc:
        codes.append('13')
        labels.append(hierarchy['13'])
        if 'sour' in desc:
            codes.extend(['15', '153'])
            labels.extend([hierarchy['15'], hierarchy['153']])
        elif 'half and half' in desc:
            codes.append('132')
            labels.append(hierarchy['132'])
        else:
            codes.append('131')
            labels.append(hierarchy['131'])
    
    elif 'dulce de leche' in desc:
        codes.extend(['15', '154'])
        labels.extend([hierarchy['15'], hierarchy['154']])
    
    # NOW check for milk, but exclude coconut milk/cream and potato products
    elif 'milk' in desc and 'coconut' not in desc and 'potato' not in desc:
        # Check for dry/dried milk FIRST before checking fat content
        if 'dry' in desc or 'dried' in desc or 'powder' in desc:
            if 'whole' in desc:
                codes.extend(['12', '121'])
                labels.extend([hierarchy['12'], hierarchy['121']])
            elif 'nonfat' in desc or 'fat free' in desc or 'skim' in desc:
                codes.extend(['12', '122'])
                labels.extend([hierarchy['12'], hierarchy['122']])
            else:
                codes.extend(['12', '123'])
                labels.extend([hierarchy['12'], hierarchy['123']])
        elif 'buttermilk' in desc and 'dried' not in desc:
            codes.extend(['11', '115'])
            labels.extend([hierarchy['11'], hierarchy['115']])
        elif 'buttermilk' in desc and 'dried' in desc:
            codes.extend(['12', '124'])
            labels.extend([hierarchy['12'], hierarchy['124']])
        elif 'evaporated' in desc or 'condensed' in desc:
            codes.extend(['15', '152'])
            labels.extend([hierarchy['15'], hierarchy['152']])
        elif 'goat' in desc:
            codes.extend(['11', '116'])
            labels.extend([hierarchy['11'], hierarchy['116']])
        elif 'whole' in desc or '3.25%' in desc:
            codes.extend(['11', '111'])
            labels.extend([hierarchy['11'], hierarchy['111']])
        elif '2%' in desc or 'reduced fat' in desc:
            codes.extend(['11', '112'])
            labels.extend([hierarchy['11'], hierarchy['112']])
        elif '1%' in desc or 'low fat' in desc or 'lowfat' in desc:
            codes.extend(['11', '113'])
            labels.extend([hierarchy['11'], hierarchy['113']])
        elif 'nonfat' in desc or 'fat free' in desc or 'skim' in desc:
            codes.extend(['11', '114'])
            labels.extend([hierarchy['11'], hierarchy['114']])
        else:
            codes.extend(['11', '117'])
            labels.extend([hierarchy['11'], hierarchy['117']])
    
    return codes, labels


def classify_meat_product(desc, hierarchy, codes, labels):
    """Classify meat, poultry, fish, and egg products"""
    
    # Check for organ meats FIRST (before animal type)
    # Use more specific patterns to avoid false matches (e.g., "tripe" in "striped")
    if (any(word in desc for word in ['liver', 'heart', 'kidney', 'brain', 'tongue', 
                                     'gizzard', 'giblet', 'variety meats', 'organ',
                                     'chitterling', 'feet', 'thymus']) or 
        ', tripe,' in desc.lower() or desc.lower().endswith(', tripe')):
        codes.append('27')
        labels.append(hierarchy['27'])
        if 'liver' in desc:
            codes.append('271')
            labels.append(hierarchy['271'])
        else:
            codes.append('272')
            labels.append(hierarchy['272'])
        return codes, labels
    
    if 'egg' in desc:
        codes.append('28')
        labels.append(hierarchy['28'])
        if 'whole' in desc or 'fried' in desc:
            codes.append('281')
            labels.append(hierarchy['281'])
        elif 'white' in desc:
            codes.append('282')
            labels.append(hierarchy['282'])
        elif 'yolk' in desc:
            codes.append('283')
            labels.append(hierarchy['283'])
        elif 'dried' in desc:
            codes.append('281')
            labels.append(hierarchy['281'])
        else:
            codes.append('284')
            labels.append(hierarchy['284'])
    
    elif 'beef' in desc:
        codes.append('21')
        labels.append(hierarchy['21'])
        if 'ground' in desc or 'patty' in desc:
            codes.append('211')
            labels.append(hierarchy['211'])
        elif 'dried' in desc or 'cured' in desc:
            codes.append('213')
            labels.append(hierarchy['213'])
        else:
            codes.append('212')
            labels.append(hierarchy['212'])
    
    elif 'pork' in desc:
        codes.append('22')
        labels.append(hierarchy['22'])
        if 'ham' in desc:
            codes.append('222')
            labels.append(hierarchy['222'])
        elif 'bacon' in desc:
            codes.append('223')
            labels.append(hierarchy['223'])
        else:
            codes.append('221')
            labels.append(hierarchy['221'])
    
    elif 'chicken' in desc:
        codes.extend(['23', '231'])
        labels.extend([hierarchy['23'], hierarchy['231']])
    
    elif 'turkey' in desc:
        codes.extend(['23', '232'])
        labels.extend([hierarchy['23'], hierarchy['232']])
    
    elif 'duck' in desc:
        codes.extend(['23', '233', '2331'])
        labels.extend([hierarchy['23'], hierarchy['233'], hierarchy['2331']])
    
    elif any(word in desc for word in ['quail', 'pheasant', 'dove']):
        codes.extend(['23', '233', '2332'])
        labels.extend([hierarchy['23'], hierarchy['233'], hierarchy['2332']])

    elif 'crustaceans' in desc or any(word in desc for word in ['shrimp', 'crab', 'lobster', 'crayfish']):
        codes.extend(['25', '251'])
        labels.extend([hierarchy['25'], hierarchy['251']])

    elif 'fish' in desc or any(fish in desc for fish in ['salmon', 'tuna', 'cod', 'catfish', 'trout', 
                                                          'haddock', 'halibut', 'herring', 'sardine',
                                                          'anchovy', 'tilapia', 'pollock', 'perch',
                                                          'mackerel', 'sea bass', 'snapper', 'swordfish',
                                                          'whiting', 'sturgeon', 'eel', 'flounder',
                                                          'pompano', 'croaker', 'shark', 'mullet']):
        codes.append('24')
        labels.append(hierarchy['24'])
        if 'canned' in desc:
            codes.append('242')
            labels.append(hierarchy['242'])
        elif 'smoked' in desc:
            codes.append('243')
            labels.append(hierarchy['243'])
        else:
            codes.append('241')
            labels.append(hierarchy['241'])
    
    elif 'mollusks' in desc or any(word in desc for word in ['clam', 'oyster', 'mussel', 'scallop', 'snail']):
        codes.append('25')
        labels.append(hierarchy['25'])
        codes.append('252')
        labels.append(hierarchy['252'])
        if 'clam' in desc or 'mussel' in desc:
            codes.append('2521')
            labels.append(hierarchy['2521'])
        elif 'oyster' in desc:
            codes.append('2522')
            labels.append(hierarchy['2522'])
        elif 'squid' in desc or 'octopus' in desc:
            codes.append('2523')
            labels.append(hierarchy['2523'])
        else:
            codes.append('2524')
            labels.append(hierarchy['2524'])
    
    elif any(word in desc for word in ['game meat', 'deer', 'venison', 'moose', 'elk', 'bison', 
                                       'bear', 'beaver', 'raccoon', 'rabbit', 'boar', 'goat',
                                       'squirrel']):
        codes.append('26')
        labels.append(hierarchy['26'])
        if 'deer' in desc or 'venison' in desc:
            codes.append('261')
            labels.append(hierarchy['261'])
        else:
            codes.append('262')
            labels.append(hierarchy['262'])
    
    elif ('frog' in desc or 'turtle' in desc) and 'black turtle' not in desc:
        codes.extend(['26', '263'])
        labels.extend([hierarchy['26'], hierarchy['263']])
    
    elif 'bacon' in desc and 'meatless' not in desc:
        codes.extend(['22', '223'])
        labels.extend([hierarchy['22'], hierarchy['223']])
    
    elif 'lamb' in desc:
        # Lamb is its own level 2 category
        codes.append('29')
        labels.append(hierarchy['29'])
    
    elif 'veal' in desc:
        # Veal is young beef
        codes.extend(['21', '214'])
        labels.extend([hierarchy['21'], hierarchy['214']])
    
    return codes, labels


def classify_legume_nut_seed(desc, hierarchy, codes, labels):
    """Classify legumes, nuts, and seeds"""
    
    # Peanuts FIRST (before any other checks)
    if 'peanut' in desc:
        codes.extend(['32', '321'])
        labels.extend([hierarchy['32'], hierarchy['321']])
        return codes, labels
    
    # Meatless bacon goes to soy
    if 'bacon, meatless' in desc:
        codes.extend(['34', '344'])
        labels.extend([hierarchy['34'], hierarchy['344']])
        return codes, labels
    
    # Soy products
    if 'soy' in desc or 'tofu' in desc or 'miso' in desc:
        codes.append('34')
        labels.append(hierarchy['34'])
        if 'soybean' in desc:
            if 'sprouted' in desc:
                codes.append('344')
                labels.append(hierarchy['344'])
            else:
                codes.append('341')
                labels.append(hierarchy['341'])
        elif 'tofu' in desc:
            codes.append('342')
            labels.append(hierarchy['342'])
        elif 'flour' in desc or 'protein' in desc or 'isolate' in desc:
            codes.append('343')
            labels.append(hierarchy['343'])
        else:
            codes.append('344')
            labels.append(hierarchy['344'])
    
    # Other beans
    elif 'bean' in desc or 'carob' in desc:
        codes.append('31')
        labels.append(hierarchy['31'])
        codes.append('311')
        labels.append(hierarchy['311'])
        
        if 'kidney' in desc:
            codes.append('3111')
            labels.append(hierarchy['3111'])
        elif 'pinto' in desc:
            codes.append('3112')
            labels.append(hierarchy['3112'])
        elif 'black' in desc:
            codes.append('3113')
            labels.append(hierarchy['3113'])
        elif 'white' in desc or 'navy' in desc or 'great northern' in desc:
            codes.append('3114')
            labels.append(hierarchy['3114'])
        elif 'lima' in desc:
            codes.append('3115')
            labels.append(hierarchy['3115'])
        elif 'carob' in desc:
            codes.append('3117')
            labels.append(hierarchy['3117'])
        else:
            codes.append('3116')
            labels.append(hierarchy['3116'])
    
    # Coconut first
    if 'coconut' in desc:
        codes.extend(['32', '323'])
        labels.extend([hierarchy['32'], hierarchy['323']])
    
    # Check for tree nuts (including chestnuts) BEFORE checking for peas
    # This prevents "european" (which contains "pea") from matching pea category
    elif any(nut in desc for nut in ['almond', 'walnut', 'pecan', 'cashew', 'pistachio', 
                                     'hazelnut', 'macadamia', 'pine nut', 'hazelnuts', 'filberts',
                                     'chestnut', 'brazilnuts', 'brazil nut']):
        codes.append('32')
        labels.append(hierarchy['32'])
        codes.append('322')
        labels.append(hierarchy['322'])
        if 'almond' in desc:
            codes.append('3221')
            labels.append(hierarchy['3221'])
        elif 'walnut' in desc:
            codes.append('3222')
            labels.append(hierarchy['3222'])
        elif 'pecan' in desc:
            codes.append('3223')
            labels.append(hierarchy['3223'])
        elif 'cashew' in desc:
            codes.append('3224')
            labels.append(hierarchy['3224'])
        elif 'brazil' in desc:
            codes.append('3225')
            labels.append(hierarchy['3225'])
        # No else - other tree nuts go directly under 322
    
    # Peas and lentils (check AFTER nuts to avoid false matches)
    elif 'cowpea' in desc or 'blackeye' in desc or 'pigeonpea' in desc or 'broadbean' in desc:
        codes.append('311')
        labels.append(hierarchy['311'])
        codes.append('3116')  # Other beans
        labels.append(hierarchy['3116'])    
    elif 'pea' in desc or 'lentil' in desc:
        codes.append('31')
        labels.append(hierarchy['31'])
        if 'lentil' in desc:
            codes.append('313')
            labels.append(hierarchy['313'])
        elif 'pea' in desc:
            codes.append('312')
            labels.append(hierarchy['312'])
            if 'green' in desc:
                codes.append('3121')
                labels.append(hierarchy['3121'])
            elif 'split' in desc:
                codes.append('3122')
                labels.append(hierarchy['3122'])
            elif 'chickpea' in desc or 'garbanzo' in desc:
                codes.append('314')
                labels.append(hierarchy['314'])
                  
    
    elif desc.startswith('seeds,') or desc.startswith('seed,'):
        codes.append('33')
        labels.append(hierarchy['33'])
        if 'sunflower' in desc:
            codes.append('331')
            labels.append(hierarchy['331'])
        elif 'pumpkin' in desc or 'squash' in desc:
            codes.append('332')
            labels.append(hierarchy['332'])
        elif 'sesame' in desc:
            codes.append('333')
            labels.append(hierarchy['333'])
        else:
            codes.append('334')
            labels.append(hierarchy['334'])
    
    return codes, labels


def classify_grain(desc, hierarchy, codes, labels):
    """Classify grain products"""
    
    # Handle baby food cereals first
    if 'babyfood, cereal' in desc:
        codes.extend(['46', '463'])
        labels.extend([hierarchy['46'], hierarchy['463']])
        return codes, labels
    
    # Check for cream of wheat specifically
    if 'cream of wheat' in desc:
        codes.extend(['46', '461'])
        labels.extend([hierarchy['46'], hierarchy['461']])
        return codes, labels
    
    # Check for buckwheat FIRST before wheat
    if 'buckwheat' in desc:
        codes.extend(['45', '455'])
        labels.extend([hierarchy['45'], hierarchy['455']])
        return codes, labels
    
    # Check for specific grain types
    if 'wheat' in desc:
        codes.append('41')
        labels.append(hierarchy['41'])
        if 'bran' in desc or 'germ' in desc:
            codes.append('412')
            labels.append(hierarchy['412'])
        else:
            codes.append('411')
            labels.append(hierarchy['411'])
    
    elif 'rice' in desc:
        codes.append('42')
        labels.append(hierarchy['42'])
        if 'brown' in desc:
            codes.append('422')
            labels.append(hierarchy['422'])
        elif 'flour' in desc:
            codes.append('423')
            labels.append(hierarchy['423'])
        elif 'wild' in desc:
            codes.append('424')
            labels.append(hierarchy['424'])
        else:
            codes.append('421')
            labels.append(hierarchy['421'])
    
    elif 'corn' in desc or 'masa' in desc or 'hominy' in desc:
        codes.append('43')
        labels.append(hierarchy['43'])
        if 'flour' in desc or 'meal' in desc or 'masa' in desc:
            codes.append('431')
            labels.append(hierarchy['431'])
        elif 'grits' in desc:
            codes.append('432')
            labels.append(hierarchy['432'])
        elif 'starch' in desc:
            codes.append('433')
            labels.append(hierarchy['433'])
        elif 'popcorn' in desc:
            codes.append('434')
            labels.append(hierarchy['434'])
        elif 'hominy' in desc:
            codes.append('435')
            labels.append(hierarchy['435'])
    
    elif 'oat' in desc:
        codes.append('44')
        labels.append(hierarchy['44'])
        if 'flour' in desc:
            codes.append('441')
            labels.append(hierarchy['441'])
        elif 'bran' in desc:
            codes.append('442')
            labels.append(hierarchy['442'])
        else:
            codes.append('443')
            labels.append(hierarchy['443'])
    
    elif 'rye' in desc:
        codes.extend(['45', '452'])
        labels.extend([hierarchy['45'], hierarchy['452']])
    
    elif 'cereal' in desc:
        codes.append('46')
        labels.append(hierarchy['46'])
        if 'baby' in desc:
            codes.append('463')
            labels.append(hierarchy['463'])
        elif 'cream of' in desc or 'instant' in desc:
            codes.append('462')
            labels.append(hierarchy['462'])
        else:
            codes.append('461')
            labels.append(hierarchy['461'])
    
    elif 'flour' in desc:
        # Generic flour - check what type
        codes.append('45')
        labels.append(hierarchy['45'])
        if 'barley' in desc:
            codes.append('451')
            labels.append(hierarchy['451'])
        elif 'millet' in desc:
            codes.append('453')
            labels.append(hierarchy['453'])
        elif 'quinoa' in desc:
            codes.append('454')
            labels.append(hierarchy['454'])
        elif 'tapioca' in desc:
            codes.append('456')
            labels.append(hierarchy['456'])
        else:
            codes.append('457')
            labels.append(hierarchy['457'])
    
    else:
        # Other grains
        codes.append('45')
        labels.append(hierarchy['45'])
        if 'barley' in desc:
            codes.append('451')
            labels.append(hierarchy['451'])
        elif 'millet' in desc:
            codes.append('453')
            labels.append(hierarchy['453'])
        elif 'quinoa' in desc:
            codes.append('454')
            labels.append(hierarchy['454'])
        elif 'tapioca' in desc:
            codes.append('456')
            labels.append(hierarchy['456'])
        else:
            codes.append('457')
            labels.append(hierarchy['457'])
    
    return codes, labels


def classify_fruit(desc, hierarchy, codes, labels):
    """Classify fruits"""
    
    # Check for dried fruits first (but not prunes which are with plums)
    if 'dried' in desc and 'prune' not in desc:
        codes.append('58')
        labels.append(hierarchy['58'])
        if 'raisin' in desc or 'grape' in desc:
            codes.append('583')
            labels.append(hierarchy['583'])
        elif 'date' in desc or 'fig' in desc:
            codes.append('581')
            labels.append(hierarchy['581'])
        elif 'apricot' in desc or 'peach' in desc:
            codes.append('582')
            labels.append(hierarchy['582'])
        else:
            codes.append('583')
            labels.append(hierarchy['583'])
        return codes, labels
    
    # Raisins (not dried) also go to dried fruits
    if 'raisin' in desc:
        codes.extend(['58', '583'])
        labels.extend([hierarchy['58'], hierarchy['583']])
        return codes, labels
    
    # Citrus fruits
    if any(citrus in desc for citrus in ['orange', 'tangerine', 'tangelo', 'mandarin']):
        codes.extend(['51', '511'])
        labels.extend([hierarchy['51'], hierarchy['511']])
    elif 'grapefruit' in desc:
        codes.extend(['51', '512'])
        labels.extend([hierarchy['51'], hierarchy['512']])
    elif 'lemon' in desc or 'lime' in desc:
        codes.extend(['51', '513'])
        labels.extend([hierarchy['51'], hierarchy['513']])
    elif 'kumquat' in desc:
        codes.extend(['51', '514'])
        labels.extend([hierarchy['51'], hierarchy['514']])
    
    # Berries
    elif 'strawberr' in desc:
        codes.extend(['52', '521'])
        labels.extend([hierarchy['52'], hierarchy['521']])
    elif 'blueberr' in desc:
        codes.extend(['52', '522'])
        labels.extend([hierarchy['52'], hierarchy['522']])
    elif 'raspberr' in desc:
        codes.extend(['52', '523'])
        labels.extend([hierarchy['52'], hierarchy['523']])
    elif 'blackberr' in desc or 'boysenberr' in desc or 'loganberr' in desc:
        codes.extend(['52', '524'])
        labels.extend([hierarchy['52'], hierarchy['524']])
    elif 'cranberr' in desc:
        codes.extend(['52', '525'])
        labels.extend([hierarchy['52'], hierarchy['525']])
    elif 'mulberr' in desc or 'currant' in desc:
        codes.extend(['52', '526'])
        labels.extend([hierarchy['52'], hierarchy['526']])
    
    # Pome fruits (apples, pears) - but exclude applesauce
    elif 'applesauce' in desc:
        codes.extend(['53', '531'])
        labels.extend([hierarchy['53'], hierarchy['531']])
    elif 'apple' in desc and 'pineapple' not in desc:
        codes.extend(['53', '531'])
        labels.extend([hierarchy['53'], hierarchy['531']])
    elif 'pear' in desc and 'asian' not in desc:
        codes.extend(['53', '532'])
        labels.extend([hierarchy['53'], hierarchy['532']])
    elif 'pear' in desc and 'asian' in desc:
        codes.extend(['53', '533'])
        labels.extend([hierarchy['53'], hierarchy['533']])
    
    # Stone fruits
    elif 'peach' in desc or 'nectarine' in desc:
        codes.extend(['54', '541'])
        labels.extend([hierarchy['54'], hierarchy['541']])
    elif 'cherr' in desc:
        codes.extend(['54', '542'])
        labels.extend([hierarchy['54'], hierarchy['542']])
    elif 'plum' in desc or 'prune' in desc:
        codes.extend(['54', '543'])
        labels.extend([hierarchy['54'], hierarchy['543']])
    elif 'apricot' in desc:
        codes.extend(['54', '544'])
        labels.extend([hierarchy['54'], hierarchy['544']])
    elif 'litchi' in desc or 'lychee' in desc:
        codes.extend(['54', '545'])
        labels.extend([hierarchy['54'], hierarchy['545']])
    
    # Tropical fruits
    elif 'banana' in desc or 'plantain' in desc:
        codes.extend(['55', '551'])
        labels.extend([hierarchy['55'], hierarchy['551']])
    elif 'pineapple' in desc:
        codes.extend(['55', '552'])
        labels.extend([hierarchy['55'], hierarchy['552']])
    elif 'mango' in desc:
        codes.extend(['55', '553'])
        labels.extend([hierarchy['55'], hierarchy['553']])
    elif 'papaya' in desc:
        codes.extend(['55', '554'])
        labels.extend([hierarchy['55'], hierarchy['554']])
    elif any(trop in desc for trop in ['guava', 'passion', 'tamarind', 'carambola']):
        codes.extend(['55', '555'])
        labels.extend([hierarchy['55'], hierarchy['555']])
    
    # Melons
    elif 'watermelon' in desc:
        codes.extend(['56', '561'])
        labels.extend([hierarchy['56'], hierarchy['561']])
    elif 'cantaloupe' in desc:
        codes.extend(['56', '562'])
        labels.extend([hierarchy['56'], hierarchy['562']])
    elif 'honeydew' in desc:
        codes.extend(['56', '563'])
        labels.extend([hierarchy['56'], hierarchy['563']])
    elif 'melon' in desc and 'waxgourd' not in desc:  # Exclude waxgourd from melons
        codes.extend(['56', '564'])
        labels.extend([hierarchy['56'], hierarchy['564']])
    
    # Grapes (fresh only, raisins handled above)
    elif 'grape' in desc and 'dried' not in desc and 'raisin' not in desc and 'leave' not in desc:
        codes.extend(['57', '571'])
        labels.extend([hierarchy['57'], hierarchy['571']])
    elif 'grape leaves' in desc:
        codes.extend(['66', '662'])        
    
    # Other fruits (avocado and olive removed - they're vegetables)
    elif 'kiwi' in desc:
        codes.extend(['59', '593'])
        labels.extend([hierarchy['59'], hierarchy['593']])
    elif 'pomegranate' in desc:
        codes.extend(['59', '594'])
        labels.extend([hierarchy['59'], hierarchy['594']])
        labels.extend([hierarchy['59'], hierarchy['594']])
    elif 'rhubarb' in desc or 'persimmon' in desc:
        codes.extend(['59', '595'])
        labels.extend([hierarchy['59'], hierarchy['595']])
    else:
        codes.extend(['59', '595'])
        labels.extend([hierarchy['59'], hierarchy['595']])
    
    return codes, labels


def classify_vegetable(desc, hierarchy, codes, labels):
    """Classify vegetables"""
    
    # Dark green vegetables
    if any(green in desc for green in ['spinach', 'kale', 'collard', 'turnip green', 
                                       'mustard green', 'beet green', 'chard']):
        codes.append('61')
        labels.append(hierarchy['61'])
        codes.append('611')
        labels.append(hierarchy['611'])
        if 'spinach' in desc:
            codes.append('6111')
            labels.append(hierarchy['6111'])
        elif 'kale' in desc or 'collard' in desc:
            codes.append('6112')
            labels.append(hierarchy['6112'])
        elif 'lettuce' in desc and 'romaine' in desc:
            codes.append('6113')
            labels.append(hierarchy['6113'])
        else:
            codes.append('6114')
            labels.append(hierarchy['6114'])
    
    elif 'broccoli' in desc:
        codes.extend(['61', '612'])
        labels.extend([hierarchy['61'], hierarchy['612']])
    
    elif any(green in desc for green in ['lettuce', 'arugula', 'watercress', 'endive', 
                                         'escarole', 'radicchio', 'dandelion', 'cress']):
        codes.extend(['61', '611', '6114'])
        labels.extend([hierarchy['61'], hierarchy['611'], hierarchy['6114']])
    
    # Deep yellow vegetables
    elif 'carrot' in desc:
        codes.extend(['62', '621'])
        labels.extend([hierarchy['62'], hierarchy['621']])
    
    elif 'pumpkin' in desc:
        codes.extend(['62', '622'])
        labels.extend([hierarchy['62'], hierarchy['622']])
    
    elif 'sweet potato' in desc:
        codes.extend(['62', '623'])
        labels.extend([hierarchy['62'], hierarchy['623']])
    
    elif 'squash' in desc and 'winter' in desc:
        codes.extend(['62', '624'])
        labels.extend([hierarchy['62'], hierarchy['624']])
    
    # Tomatoes
    elif 'tomato' in desc:
        codes.append('63')
        labels.append(hierarchy['63'])
        if 'raw' in desc:
            codes.append('631')
            labels.append(hierarchy['631'])
        elif 'cooked' in desc or 'stewed' in desc:
            codes.append('632')
            labels.append(hierarchy['632'])
        elif 'paste' in desc:
            codes.extend(['633', '6331'])
            labels.extend([hierarchy['633'], hierarchy['6331']])
        elif 'sauce' in desc:
            codes.extend(['633', '6332'])
            labels.extend([hierarchy['633'], hierarchy['6332']])
        elif 'puree' in desc:
            codes.extend(['633', '6333'])
            labels.extend([hierarchy['633'], hierarchy['6333']])
        else:
            codes.append('633')
            labels.append(hierarchy['633'])
    
    # Potatoes
    elif 'potato' in desc and 'sweet' not in desc:
        codes.append('64')
        labels.append(hierarchy['64'])
        if 'canned' in desc:
            codes.append('642')
            labels.append(hierarchy['642'])
        elif 'dehydrated' in desc or 'flakes' in desc or 'mashed' in desc:
            codes.append('643')
            labels.append(hierarchy['643'])
        elif 'flour' in desc:
            codes.append('644')
            labels.append(hierarchy['644'])
        else:
            codes.append('641')
            labels.append(hierarchy['641'])
    
    # Other starchy vegetables
    elif 'cassava' in desc or 'yuca' in desc:
        codes.extend(['65', '654'])
        labels.extend([hierarchy['65'], hierarchy['654']])
    
    elif 'taro' in desc or 'yam' in desc or 'jicama' in desc:
        codes.extend(['65', '655'])
        labels.extend([hierarchy['65'], hierarchy['655']])
    
    # Snap beans and edible pods
    elif 'beans, snap' in desc or 'edible-podded' in desc:
        codes.extend(['66', '661', '6614'])
        labels.extend([hierarchy['66'], hierarchy['661'], hierarchy['6614']])
    
    # Other vegetables
    elif 'onion' in desc or 'scallion' in desc or 'leek' in desc or 'garlic' in desc or 'chive' in desc:
        codes.append('66')
        labels.append(hierarchy['66'])
        codes.append('661')
        labels.append(hierarchy['661'])
        codes.append('6610')
        labels.append(hierarchy['6610'])        
        if 'onion' in desc:
            codes.append('66100')
            labels.append(hierarchy['66100'])
        elif 'garlic' in desc:
            codes.append('66101')
            labels.append(hierarchy['66101'])
        elif 'leek' in desc or 'scallion' in desc:
            codes.append('66102')
            labels.append(hierarchy['66102'])
    
    elif 'cucumber' in desc or 'squash' in desc or 'waxgourd' in desc:
        codes.append('66')
        labels.append(hierarchy['66'])
        codes.append('661')
        labels.append(hierarchy['661'])
        codes.append('6611')
        labels.append(hierarchy['6611'])        
        if 'cucumber' in desc:
            codes.append('66110')
            labels.append(hierarchy['66110'])
        else:
            codes.append('66111')
            labels.append(hierarchy['66111'])
    
    elif any(cruci in desc for cruci in ['cabbage', 'cauliflower', 'brussels sprout', 
                                         'kohlrabi', 'rutabaga', 'turnip', 'artichoke', 'asparagus']):
        codes.append('66')
        labels.append(hierarchy['66'])
        codes.append('661')
        labels.append(hierarchy['661'])
        codes.append('6612')
        labels.append(hierarchy['6612'])        
        if 'cabbage' in desc:
            codes.append('66120')
            labels.append(hierarchy['66120'])
        elif 'cauliflower' in desc:
            codes.append('66121')
            labels.append(hierarchy['66121'])
        elif 'brussels' in desc:
            codes.append('66122')
            labels.append(hierarchy['66122'])
        elif 'artichoke' in desc:
            codes.append('66123')
            labels.append(hierarchy['66123'])
        elif 'asparagus' in desc:
            codes.append('66124')
            labels.append(hierarchy['66124'])
        elif 'rutabaga' in desc:
            codes.append('66125')
            labels.append(hierarchy['66125'])
        elif 'turnip' in desc:
            codes.append('66126')
            labels.append(hierarchy['66126'])

    elif 'mushroom' in desc:
        codes.extend(['66', '661', '6613'])
        labels.extend([hierarchy['66'], hierarchy['661'], hierarchy['6613']])
    
    # Peppers (sweet and hot)
    elif 'pepper' in desc:
        codes.append('66')
        labels.append(hierarchy['66'])
        codes.append('661')
        labels.append(hierarchy['661'])
        if 'hot' in desc or 'chili' in desc:
            codes.append('66151')
            labels.append(hierarchy['66151'])
        elif 'sweet' in desc:
            codes.append('66150')
            labels.append(hierarchy['66150'])
        else:
            codes.extend(['95', '951'])
            labels.extend([hierarchy['95'], hierarchy['951']])
    
    # Seaweed
    elif 'seaweed' in desc or any(sea in desc for sea in ['agar', 'kelp', 'laver', 'wakame', 'spirulina']):
        codes.extend(['66', '661', '6616'])
        labels.extend([hierarchy['66'], hierarchy['661'], hierarchy['6616']])
    
    # Avocados
    elif 'avocado' in desc:
        codes.extend(['66', '661', '6617'])
        labels.extend([hierarchy['66'], hierarchy['661'], hierarchy['6617']])
    
    # Olives
    elif 'olive' in desc and 'oil' not in desc:
        codes.extend(['66', '661', '6618'])
        labels.extend([hierarchy['66'], hierarchy['661'], hierarchy['6618']])
    
    # Balsam-pear, lotus, pokeberry, waterchestnut...etc
    elif any(other in desc for other in ['balsam-pear', 'lotus', 'pokeberry', 'waterchestnut', 'tomatillo', 'alfalfa', 'capers']):
        codes.extend(['66', '662'])
        labels.extend([hierarchy['66'], hierarchy['662']])
    
    else:
        codes.extend(['66', '662'])
        labels.extend([hierarchy['66'], hierarchy['662']])
    
    return codes, labels


def classify_fat_oil(desc, hierarchy, codes, labels):
    """Classify fats and oils"""
    
    # Butter
    if 'butter' in desc:
        codes.append('73')
        labels.append(hierarchy['73'])
        return codes, labels
    
    # Vegetable oils
    if 'oil' in desc and 'pea' not in desc:
        codes.append('71')
        labels.append(hierarchy['71'])
        if 'olive' in desc:
            codes.extend(['712', '7121'])
            labels.extend([hierarchy['712'], hierarchy['7121']])
        elif 'corn' in desc:
            codes.extend(['712', '7122'])
            labels.extend([hierarchy['712'], hierarchy['7122']])
        elif 'canola' in desc:
            codes.extend(['712', '7123'])
            labels.extend([hierarchy['712'], hierarchy['7123']])
        elif 'vegetable' in desc:
            codes.append('711')
            labels.append(hierarchy['711'])
        else:
            codes.extend(['712', '7124'])
            labels.extend([hierarchy['712'], hierarchy['7124']])
    
    # Animal fats
    elif any(fat in desc for fat in ['fat,', 'grease', 'lard']):
        codes.append('72')
        labels.append(hierarchy['72'])
        if 'bacon' in desc or 'pork' in desc:
            codes.append('721')
            labels.append(hierarchy['721'])
        elif 'chicken' in desc:
            codes.append('722')
            labels.append(hierarchy['722'])
        elif 'turkey' in desc:
            codes.append('723')
            labels.append(hierarchy['723'])
        else:
            codes.append('724')
            labels.append(hierarchy['724'])
    
    return codes, labels


def classify_sugar_beverage(desc, hierarchy, codes, labels):
    """Classify sugars, sweets, and beverages"""
    
    # Alcoholic beverages
    if any(alc in desc for alc in ['alcoholic beverage', 'wine', 'beer', 'liqueur', 'sake', 'cider', 'hard cider']):
        codes.append('80')
        labels.append(hierarchy['80'])
        if 'distilled' in desc or 'gin' in desc or 'rum' in desc or 'vodka' in desc or 'whiskey' in desc:
            codes.append('801')
            labels.append(hierarchy['801'])
        elif 'wine' in desc:
            codes.append('802')
            labels.append(hierarchy['802'])
        elif 'beer' in desc or 'cider' in desc or 'amber' in desc:
            codes.append('803')
            labels.append(hierarchy['803'])
        elif 'liqueur' in desc:
            codes.append('804')
            labels.append(hierarchy['804'])
        elif 'sake' in desc:
            codes.append('805')
            labels.append(hierarchy['805'])
        return codes, labels
    
    if 'sugar' in desc:
        codes.append('81')
        labels.append(hierarchy['81'])
        if 'white' in desc or 'granulated' in desc:
            codes.append('811')
            labels.append(hierarchy['811'])
        else:
            codes.append('812')
            labels.append(hierarchy['812'])
    
    elif 'syrup' in desc:
        codes.append('82')
        labels.append(hierarchy['82'])
        if 'maple' in desc:
            codes.append('821')
            labels.append(hierarchy['821'])
        elif 'corn' in desc:
            codes.append('822')
            labels.append(hierarchy['822'])
        elif 'agave' in desc:
            codes.append('825')
            labels.append(hierarchy['825'])
        else:
            codes.append('826')
            labels.append(hierarchy['826'])
    
    elif 'honey' in desc:
        codes.extend(['82'])
        labels.extend([hierarchy['82']])
    
    elif 'molasses' in desc:
        codes.extend(['82'])
        labels.extend([hierarchy['82']])
    
    elif 'gelatin' in desc:
        codes.append('83')
        labels.append(hierarchy['83'])
        codes.append('830')
        labels.append(hierarchy['830'])
    
    elif 'sweetener' in desc:
        codes.extend(['82', '826'])
        labels.extend([hierarchy['82'], hierarchy['826']])
    
    elif 'coffee' in desc:
        codes.extend(['87', '871'])
        labels.extend([hierarchy['87'], hierarchy['871']])

    elif 'tea' in desc:
        codes.extend(['87', '872'])
        labels.extend([hierarchy['87'], hierarchy['872']])

    elif 'water' in desc:
        codes.append('84')
        labels.append(hierarchy['84'])
        if 'tap' in desc:
            codes.append('841')
            labels.append(hierarchy['841'])
        elif 'bottled' in desc:
            codes.append('842')
            labels.append(hierarchy['842'])
        elif 'coconut' in desc:
            # Coconut water should not be here (handled by nuts)
            pass
        else:
            codes.append('841')
            labels.append(hierarchy['841'])
    
    elif 'whey' in desc and 'beverage' in desc:
        codes.extend(['88', '881'])
        labels.extend([hierarchy['88'], hierarchy['881']])
    
    elif 'cocoa' in desc:
        codes.extend(['89', '891'])
        labels.extend([hierarchy['89'], hierarchy['891']])
    
    elif 'chocolate' in desc:
        codes.extend(['89', '892'])
        labels.extend([hierarchy['89'], hierarchy['892']])
    
    elif 'gelatin' in desc:
        codes.extend(['83'])
        labels.extend([hierarchy['83']])
    
    elif 'snacks' in desc:
        codes.append('83')
        labels.append(hierarchy['83'])
    
    return codes, labels


def classify_seasoning(desc, hierarchy, codes, labels):
    """Classify seasonings, condiments, and leavening agents"""
    
    if 'salt' in desc:
        codes.append('91')
        labels.append(hierarchy['91'])
    
    elif 'spices' in desc or desc.startswith('spice'):
        codes.append('92')
        labels.append(hierarchy['92'])
        if any(seed in desc for seed in ['seed', 'mustard seed']):
            codes.append('922')
            labels.append(hierarchy['922'])
        else:
            codes.append('921')
            labels.append(hierarchy['921'])
    
    elif any(herb in desc for herb in ['basil', 'oregano', 'thyme', 'parsley', 'cilantro', 
                                       'coriander', 'sage', 'marjoram']):
        codes.append('93')
        labels.append(hierarchy['93'])
        if 'dried' in desc or 'ground' in desc:
            codes.append('931')
            labels.append(hierarchy['931'])
        else:
            codes.append('932')
            labels.append(hierarchy['932'])
    
    elif 'vinegar' in desc:
        codes.append('94')
        labels.append(hierarchy['94'])
    
    elif 'sauce' in desc:
        codes.append('95')
        labels.append(hierarchy['95'])
        if 'hot' in desc or 'tabasco' in desc or 'pepper' in desc:
            codes.append('951')
            labels.append(hierarchy['951'])
        elif 'cranberr' in desc:
            # cranberry sauce should not be here
            codes.append('525')  
        elif 'guava' in desc:
            # guava sauce should not be here
            codes.append('555')              
        else:
            codes.append('952')
            labels.append(hierarchy['952'])
    
    elif 'leavening' in desc or 'yeast' in desc:
        codes.append('96')
        labels.append(hierarchy['96'])
        if 'yeast' in desc and 'extract' not in desc:
            codes.append('961')
            labels.append(hierarchy['961'])
        elif 'baking powder' in desc:
            codes.append('962')
            labels.append(hierarchy['962'])
        elif 'baking soda' in desc:
            codes.append('963')
            labels.append(hierarchy['963'])
        else:
            codes.append('964')
            labels.append(hierarchy['964'])
    
    elif 'extract' in desc or 'vanilla' in desc:
        codes.append('97')
        labels.append(hierarchy['97'])
    
    elif 'yeast extract' in desc:
        codes.extend(['96', '964'])
        labels.extend([hierarchy['96'], hierarchy['964']])
    
    return codes, labels


def main():
    """Main execution function"""
    print("=" * 70)
    print("Building Ingredient Hierarchy Taxonomy")
    print("=" * 70)
    
    # Step 1: Load and categorize ingredients
    print("\nStep 1: Loading and categorizing ingredients...")
    ingredients_df = load_ingredients()
    print(f"Loaded {len(ingredients_df)} ingredients")
    
    # Step 2: Build hierarchy structure
    print("\nStep 2: Building hierarchy structure...")
    hierarchy = build_hierarchy(ingredients_df)
    print(f"Created {len(hierarchy)} hierarchical nodes")
    
    # Step 3: Assign hierarchical codes to ingredients
    print("\nStep 3: Assigning hierarchical codes to ingredients...")
    mapping_df = assign_hierarchical_codes(ingredients_df, hierarchy)
    print(f"Assigned codes to {len(mapping_df)} ingredients")
    
    # Step 4: Save outputs
    print("\nStep 4: Saving outputs...")
    
    # Step 4a: Add each ingredient as a leaf node in the hierarchy
    print("  - Adding ingredient leaf nodes...")
    
    # Group ingredients by their hierarchical code
    from collections import defaultdict
    ingredients_by_code = defaultdict(list)
    for _, row in mapping_df.iterrows():
        ingredients_by_code[row['hierarchical_code']].append(row)
    
    # Add ingredients as sequential children of their parent categories
    for parent_code in sorted(ingredients_by_code.keys(), key=lambda x: str(x)):
        ingredients = ingredients_by_code[parent_code]
        # Sort ingredients by their original code for consistency
        ingredients = sorted(ingredients, key=lambda x: x['original_code'])
        
        # Handle categories with more than 9 ingredients by creating subcategories
        if len(ingredients) > 9:
            # Create intermediate subcategory level
            # For each group of 10 ingredients, create a subcategory
            num_subcategories = (len(ingredients) + 9) // 10  # Ceiling division
            
            for subcat_idx in range(num_subcategories):
                # Create subcategory code (4-digit)
                subcat_code = f"{parent_code}{subcat_idx}"
                subcat_label = f"{hierarchy[parent_code]}_subcategory_{subcat_idx + 1}"
                hierarchy[subcat_code] = subcat_label
                
                # Assign ingredients to this subcategory (up to 10 ingredients)
                start_idx = subcat_idx * 10
                end_idx = min(start_idx + 10, len(ingredients))
                
                for local_idx, ing_idx in enumerate(range(start_idx, end_idx)):
                    ing = ingredients[ing_idx]
                    # Create 5-digit leaf code under subcategory
                    leaf_code = f"{subcat_code}{local_idx}"
                    hierarchy[leaf_code] = ing['ingredient_description']
                    # Store the leaf code mapping for reference
                    mapping_df.loc[mapping_df['original_code'] == ing['original_code'], 'leaf_code'] = leaf_code
        else:
            # For categories with 9 or fewer ingredients, use simple sequential numbering
            for idx, ing in enumerate(ingredients):
                # Create leaf code by adding one digit (4-digit code)
                leaf_code = f"{parent_code}{idx}"
                hierarchy[leaf_code] = ing['ingredient_description']
                # Store the leaf code mapping for reference
                mapping_df.loc[mapping_df['original_code'] == ing['original_code'], 'leaf_code'] = leaf_code
    
    # Step 4b: Convert all codes to strings and sort
    print("  - Sorting hierarchy...")
    hierarchy_sorted = OrderedDict()
    
    # Custom sort function to handle numeric sorting properly
    def sort_key(code_str):
        # Pad each numeric segment to ensure proper sorting
        # e.g., "51" -> "051", "510" -> "510", "5100" -> "5100"
        parts = []
        current = ""
        for char in str(code_str):
            if char.isdigit():
                current += char
            else:
                if current:
                    parts.append(current.zfill(4))
                    current = ""
                parts.append(char)
        if current:
            parts.append(current.zfill(4))
        return ''.join(parts)
    
    # Sort by the custom key
    for code in sorted(hierarchy.keys(), key=sort_key):
        hierarchy_sorted[str(code)] = hierarchy[code]
    
# Save the hierarchy taxonomy (similar to NodeLabelsMCT.txt format)
    with open('../../data/00/food_tree/ingredient_tree.txt', 'w') as f:
        f.write("Level.code\tIngredient.category.description\n")
        # Sort keys as strings
        for code in sorted(hierarchy_sorted.keys(), key=str):
            f.write(f"{code}\t{hierarchy_sorted[code]}\n")
    print("  - Saved ingredient_tree.txt")
    
    # Save the mapping from original codes to hierarchical codes (including leaf codes)
    mapping_df.to_csv('../../data/00/ingredient_code_mapping.csv', index=False)
    print("  - Saved ingredient_code_mapping.csv")
    
    print("\n" + "=" * 70)
    print("Files created successfully!")
    print("=" * 70)
    print("\nOutput files:")
    print("  1. ingredient_tree.txt - Hierarchical taxonomy structure")
    print("  2. ingredient_code_mapping.csv - Mapping from original to hierarchical codes")
    
    return mapping_df, hierarchy


if __name__ == "__main__":
    mapping_df, hierarchy = main()